# Trabajo Práctico 2: Armado de un esquema de aprendizaje automático

En el Trabajo Práctico final se espera que puedan poner en práctica los conocimientos adquiridos en el curso, trabajando con un conjunto de datos de clasificación.

El objetivo es que se introduzcan en el desarrollo de un esquema para hacer tareas de aprendizaje automático: selección de un modelo, ajuste de hiperparámetros y evaluación.

El conjunto de datos a utilizar está en `./data/loan_data.csv`.   
El conjunto de datos a utilizar está en `("https://raw.githubusercontent.com/DiploDatos/IntroduccionAprendizajeAutomatico/master/data/loan_data.csv", comment="#")`.

Si abren el archivo verán que al principio (las líneas que empiezan con `#`) describen el conjunto de datos y sus atributos (incluyendo el atributo de etiqueta o clase).

Se espera que hagan uso de las herramientas vistas en el curso. Se espera que hagan uso especialmente de las herramientas brindadas por `scikit-learn`.

# Orientación general del Trabajo Práctico 2

En este trabajo práctico vamos a resolver un problema de **clasificación supervisada**.

El objetivo no es solamente entrenar modelos, sino recorrer el flujo completo de trabajo:

1. Comprender el problema y el dataset.
2. Identificar la variable objetivo.
3. Analizar si las clases están balanceadas.
4. Separar datos de entrenamiento y evaluación.
5. Entrenar modelos de clasificación.
6. Ajustar hiperparámetros con validación cruzada.
7. Evaluar los modelos con métricas apropiadas.
8. Comparar modelos y justificar una recomendación final.

A lo largo del trabajo vamos a intentar responder una pregunta central:

> ¿Qué modelo recomendaríamos usar para este problema y con qué evidencia lo justificaríamos?


# Evaluación con métricas

Para cada modelo evaluado deberíamos reportar, como mínimo:

- accuracy;
- precision;
- recall;
- F1-score;
- matriz de confusión.

Pero además debemos interpretar esos números.

Preguntas orientadoras:

1. ¿El modelo clasifica igual de bien ambas clases?
2. ¿Hay muchos falsos positivos?
3. ¿Hay muchos falsos negativos?
4. ¿Qué métrica parece más relevante para este problema?
5. ¿La accuracy alcanza para decidir o necesitamos mirar otras métricas?

Recordemos que, en problemas con clases desbalanceadas, la accuracy puede ocultar errores importantes.


# Análisis exploratorio mínimo

Antes de entrenar modelos, revisemos:

1. Tamaño del dataset.
2. Nombre y tipo de las variables.
3. Valores faltantes.
4. Distribución de la variable `TARGET`.
5. Posible desbalance de clases.

En particular, para `TARGET` deberíamos calcular:

```python
df["TARGET"].value_counts()
df["TARGET"].value_counts(normalize=True)
```

Si una clase aparece mucho más que la otra, la accuracy puede ser engañosa. En ese caso, deberemos mirar también precision, recall, F1-score y matriz de confusión.


## Antes de empezar: costo del error

Como el problema está relacionado con la aprobación de préstamos o créditos, no todos los errores tienen el mismo significado.

Conviene pensar desde el comienzo:

- **Falso positivo:** el modelo recomienda aprobar un préstamo que no debería aprobarse.
- **Falso negativo:** el modelo recomienda rechazar un préstamo que sí debería aprobarse.

Preguntas para tener presentes durante todo el trabajo:

- ¿Cuál de estos errores sería más costoso para el banco?
- ¿Cuál sería más perjudicial para el cliente?
- ¿Qué métrica nos ayudaría a controlar mejor cada tipo de error?


Al evaluar el riesgo crediticio en el desarrollo de este modelo, es fundamental traducir las métricas a su significado real en el contexto de la aprobación de préstamos.

**1. ¿Cuál de estos errores sería más costoso para el banco?**
El error más destructivo financieramente para la entidad es el **Falso Positivo**, donde el modelo recomienda aprobar un préstamo que no debería aprobarse. Al analizar la viabilidad de créditos, bonos o cualquier instrumento de renta fija, la pérdida irrecuperable del capital principal prestado impacta mucho más fuerte en los flujos de caja y el balance de la institución que el costo de oportunidad (dejar de percibir intereses por rechazar erróneamente a un buen cliente).

**2. ¿Cuál sería más perjudicial para el cliente?**
Para el solicitante, el escenario más perjudicial es el **Falso Negativo**, donde el modelo recomienda rechazar un préstamo que sí debería aprobarse. En este caso, un cliente con la solvencia adecuada ve denegado el acceso al crédito, truncando sus necesidades de financiamiento y sus proyectos personales o comerciales de forma completamente injustificada.

**3. ¿Qué métrica nos ayudaría a controlar mejor cada tipo de error?**
Dado que en el conjunto de datos la variable objetivo identifica si el cliente entró en morosidad (`TARGET = 1`) o si cumplió de forma regular (`TARGET = 0`), la métrica de *accuracy* no alcanza para tomar decisiones de negocio y necesitamos mirar el desglose del desempeño.

* **Para controlar la aprobación de préstamos riesgosos (Falsos Positivos del enunciado):** Si el objetivo principal es proteger los fondos del banco, debemos maximizar el **Recall (Exhaustividad) de la clase 1 (Default)**. Un recall alto asegura que el modelo "atrape" e identifique a la mayor cantidad posible de clientes riesgosos antes de que se apruebe el dinero. Visto desde la clase opuesta, esto equivale a buscar una alta **Precisión (*Precision*) en la clase 0**, lo cual nos garantiza que cuando el sistema emite una orden de "Aprobar", la probabilidad de pago es extremadamente alta.
* **Para controlar los rechazos injustos (Falsos Negativos del enunciado):** Si se busca proteger la experiencia del cliente y no frenar el otorgamiento crediticio, el objetivo sería maximizar la **Precisión de la clase 1**. Una precisión alta aquí significa que, al catalogar a un solicitante como "moroso" para rechazarlo, el modelo está sumamente seguro de esa predicción, evitando así denegar solicitudes a clientes sanos. En paralelo, esto se refleja en un alto **Recall de la clase 0**, asegurando que logramos captar y aprobar a la enorme mayoría de los clientes aptos.

Dado el desbalance de clases habitual en las carteras crediticias, el análisis detallado de la **Matriz de Confusión** y la comparación de modelos mediante el **F1-score** resultarán claves para justificar qué punto de equilibrio entre riesgo y aprobación masiva se recomienda finalmente.

In [1]:
import numpy as np
import pandas as pd

# TODO: Agregar las librerías que hagan falta
from sklearn.model_selection import train_test_split

## Recomendación sobre la partición Train/Test

Como estamos trabajando con un problema de clasificación, conviene que la proporción de clases sea parecida en entrenamiento y test.

Para eso podemos usar:

```python
train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

El argumento `stratify=y` ayuda a conservar la proporción de clases en ambas particiones.


## Carga de datos y división en entrenamiento y evaluación

La celda siguiente se encarga de la carga de datos (haciendo uso de pandas). Estos serán los que se trabajarán en el resto del laboratorio.

In [2]:
#dataset = pd.read_csv("./data/loan_data.csv", comment="#")
dataset = pd.read_csv("https://raw.githubusercontent.com/DiploDatos/IntroduccionAprendizajeAutomatico/master/data/loan_data.csv", comment="#")

dataset.to_csv("loan_data.csv", index=False)


# División entre instancias y etiquetas
X, y = dataset.iloc[:, 1:], dataset.TARGET

# división entre entrenamiento y evaluación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [11]:
print(X, y)

       LOAN   MORTDUE     VALUE   YOJ  DEROG  DELINQ       CLAGE  NINQ  CLNO  \
0      4700   88026.0  115506.0   6.0    0.0     0.0  182.248332   0.0  27.0   
1     19300   39926.0  101208.0   4.0    0.0     0.0  140.051638   0.0  14.0   
2      5700   71556.0   79538.0   2.0    0.0     0.0   92.643085   0.0  15.0   
3     13000   44875.0   57713.0   0.0    1.0     0.0  184.990324   1.0  12.0   
4     19300   72752.0  106084.0  11.0    0.0     0.0  193.707100   1.0  13.0   
...     ...       ...       ...   ...    ...     ...         ...   ...   ...   
1849  53400  228236.0  305514.0   6.0    0.0     0.0   11.148069   0.0   2.0   
1850  53600  235895.0  299772.0   5.0    0.0     0.0  112.748282   7.0  22.0   
1851  53600  208197.0  297280.0   4.0    1.0     1.0  160.485251   2.0  29.0   
1852  65500  205156.0  290239.0   2.0    0.0     0.0   98.808206   1.0  21.0   
1853  77400   87651.0  224630.0   9.0    0.0     2.0   73.469630   3.0  13.0   

         DEBTINC  
0      29.209023  
1


Documentación:

- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

## Ejercicio 1: Descripción de los Datos y la Tarea

Responder las siguientes preguntas:

1. ¿De qué se trata el conjunto de datos?
2. ¿Cuál es la variable objetivo que hay que predecir? ¿Qué significado tiene?
3. ¿Qué información (atributos) hay disponible para hacer la predicción?
4. ¿Qué atributos imagina ud. que son los más determinantes para la predicción?

**No hace falta escribir código para responder estas preguntas.**

**1. ¿De qué se trata el conjunto de datos?**
El conjunto de datos contiene información histórica sobre solicitudes de préstamos o créditos (generalmente hipotecarios). Recopila características del perfil financiero de distintos clientes y su historial crediticio al momento de la solicitud, con el propósito de evaluar el nivel de riesgo asociado a cada operación.

**2. ¿Cuál es la variable objetivo que hay que predecir? ¿Qué significado tiene?**
La variable objetivo es **`TARGET`**, la cual es una etiqueta binaria que toma los valores `0` o `1`.
Su significado central es identificar la cobrabilidad del crédito. Habitualmente, el valor `1` indica que el cliente entró en morosidad, incumplió con los pagos (hizo default) o requirió acciones de cobranza severas. El valor `0` significa que el cliente cumplió con sus obligaciones de pago de forma regular. El objetivo del modelo será predecir si un nuevo solicitante tiene perfil de clase 1 o de clase 0 para ayudar al banco a aprobar o rechazar la solicitud.

**3. ¿Qué información (atributos) hay disponible para hacer la predicción?**
Al observar el bloque de datos de la celda ejecutada en tu notebook, contamos con atributos puramente cuantitativos que reflejan el estado patrimonial y el comportamiento financiero del cliente:

* **LOAN:** Monto total del préstamo que el cliente está solicitando.
* **MORTDUE:** Monto adeudado en la hipoteca existente.
* **VALUE:** Valor de tasación o mercado de la propiedad actual.
* **YOJ:** Años de antigüedad en el empleo actual (mide la estabilidad laboral).
* **DEROG:** Cantidad de reportes crediticios negativos o derogatorios mayores.
* **DELINQ:** Cantidad de líneas de crédito que actualmente se encuentran en estado de morosidad.
* **CLAGE:** Antigüedad de la línea de crédito más vieja del cliente (en meses).
* **NINQ:** Número de consultas recientes al historial crediticio del cliente (muchas consultas pueden indicar búsqueda desesperada de liquidez).
* **CLNO:** Cantidad total de líneas de crédito que posee el cliente.
* **DEBTINC:** Relación deuda-ingreso (*Debt-to-income ratio*), que mide qué porcentaje de los ingresos se destina a pagar deudas.

**4. ¿Qué atributos imagina Ud. que son los más determinantes para la predicción?**
Al evaluar el riesgo crediticio y la solidez de los flujos de caja futuros, existen variables que estructuralmente pesan más que otras. Los atributos más determinantes probablemente sean:

* **DEBTINC (Relación deuda-ingreso):** Es el indicador más directo de la capacidad de pago. Si el porcentaje de ingresos que el cliente ya tiene comprometido en deudas es muy alto, su flujo de efectivo mensual será demasiado estrecho, aumentando enormemente el riesgo de impago ante cualquier eventualidad.
* **DELINQ y DEROG:** Son los reflejos del comportamiento de pago histórico. Contar con un registro de líneas ya atrasadas o informes derogatorios es una de las alertas de riesgo más fuertes, ya que el comportamiento pasado suele ser el mejor predictor del futuro en instrumentos de crédito.
* **VALUE en relación a MORTDUE y LOAN:** El grado de apalancamiento que tiene el cliente. Si el valor de la propiedad (`VALUE`) apenas cubre lo que ya debe de hipoteca (`MORTDUE`) más el nuevo préstamo (`LOAN`), el banco carece de una garantía sólida en caso de tener que liquidar el activo, lo que eleva el perfil de riesgo de la transacción.

EL Texto comentado al inicio del Dataset es el siguiente:


 Loan dataset based on the Kaggle Home Equity dataset
 Available at: https://www.kaggle.com/ajay1735/hmeq-data

 Context
 =======
 The consumer credit department of a bank wants to automate the decisionmaking
 process for approval of home equity lines of credit. To do this, they will
 follow the recommendations of the Equal Credit Opportunity Act to create an
 empirically derived and statistically sound credit scoring model. The model
 will be based on data collected from recent applicants granted credit through
 the current process of loan underwriting. The model will be built from
 predictive modeling tools, but the created model must be sufficiently
 interpretable to provide a reason for any adverse actions (rejections).

 Content
 =======

 The Home Equity dataset (HMEQ) contains baseline and loan performance
 information for 5,960 recent home equity loans.

 The target (BAD) is a binary variable indicating whether an applicant eventually defaulted or was seriously delinquent. This adverse outcome occurred in 1,189 cases (20%).

 For each applicant, 12 input variables were recorded.

 Attributes
 ==========
 Name    Description

 TARGET  Label: 1 = client defaulted on loan - 0 = loan repaid

 LOAN    Amount of the loan request

 MORTDUE Amount due on existing mortgage

 VALUE   Value of current property

 YOJ     Years at present job

 DEROG   Number of major derogatory reports

 DELINQ  Number of delinquent credit lines

 CLAGE   Age of oldest trade line in months

 NINQ    Number of recent credit lines

 CLNO    Number of credit lines

 DEBTINC Debt-to-income ratio


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- REEMPLAZA 'df' POR EL NOMBRE DE TU DATAFRAME SI ES DISTINTO ---
# data = df.copy()

print("="*50)
print("1. CARACTERÍSTICAS PRINCIPALES Y TIPOS DE DATOS")
print("="*50)
# info() muestra las columnas, tipos de datos (dtypes) y la memoria usada
dataset.info()

print("\n--- Estadísticas Descriptivas (Numéricas) ---")
display(dataset.describe())

print("\n" + "="*50)
print("2. LISTADO DE VALORES NULOS (NA)")
print("="*50)
# Sumamos los nulos por columna y mostramos solo las que tienen más de 0
nulos = dataset.isna().sum()
nulos_filtrados = nulos[nulos > 0].sort_values(ascending=False)

if not nulos_filtrados.empty:
    print("Columnas con valores NA:")
    print(nulos_filtrados)
    print("\nPorcentaje de NA sobre el total:")
    print((nulos_filtrados / len(dataset) * 100).round(2).astype(str) + ' %')
else:
    print("¡No se encontraron valores nulos (NA) en el DataFrame!")

print("\n" + "="*50)
print("3. BÚSQUEDA DE CEROS")
print("="*50)
# Buscamos cuántas veces aparece el número 0 exacto en cada columna numérica
# Primero filtramos solo las columnas numéricas para evitar errores
dataset_numeric = dataset.select_dtypes(include=[np.number])
ceros = (dataset_numeric == 0).sum()
ceros_filtrados = ceros[ceros > 0].sort_values(ascending=False)

if not ceros_filtrados.empty:
    print("Columnas que contienen valores iguales a 0:")
    print(ceros_filtrados)
else:
    print("¡No se encontraron valores iguales a 0 en las columnas numéricas!")

print("\n" + "="*50)
print("4. VISUALIZACIÓN DE DISTRIBUCIONES (GAUSSIANAS / NORMALES)")
print("="*50)

# Configuramos el lienzo de Matplotlib dinámicamente según la cantidad de columnas
columnas_num = dataset_numeric.columns
num_cols = len(columnas_num)
filas = (num_cols // 3) + (1 if num_cols % 3 != 0 else 0) # 3 gráficos por fila

fig, axes = plt.subplots(nrows=filas, ncols=3, figsize=(15, 4 * filas))
axes = axes.flatten() # Aplanamos la matriz de gráficos para iterar fácilmente

# Colores estéticos para los gráficos
colores = sns.color_palette("husl", num_cols)

for i, col in enumerate(columnas_num):
    # sns.histplot con kde=True dibuja el histograma y la curva de campana
    sns.histplot(dataset[col].dropna(), kde=True, ax=axes[i], color=colores[i], bins=30)
    axes[i].set_title(f'Distribución de: {col}', fontweight='bold')
    axes[i].set_ylabel('Frecuencia')

# Ocultar los subplots vacíos (si la cantidad de columnas no es múltiplo de 3)
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## Preguntas orientadoras para el análisis del dataset

Antes de entrenar cualquier modelo, respondamos con texto:

1. ¿De qué se trata este conjunto de datos?
2. ¿Qué representa la variable `TARGET`?
3. ¿Qué significa la clase `0` y qué significa la clase `1`?
4. ¿Cuál es el problema que intenta resolver el banco?
5. ¿Qué variables predictoras tenemos disponibles?
6. ¿Qué variables creemos que podrían ser más importantes?
7. ¿Hay variables que podrían estar relacionadas entre sí?
8. ¿Qué información adicional nos gustaría tener para comprender mejor el problema?

La idea es no empezar directamente por el modelo. Primero necesitamos comprender el problema.


Header del dataset:

 Loan dataset based on the Kaggle Home Equity dataset
 Available at: https://www.kaggle.com/ajay1735/hmeq-data

 Context
 =======
 The consumer credit department of a bank wants to automate the decisionmaking
 process for approval of home equity lines of credit. To do this, they will
 follow the recommendations of the Equal Credit Opportunity Act to create an
 empirically derived and statistically sound credit scoring model. The model
 will be based on data collected from recent applicants granted credit through
 the current process of loan underwriting. The model will be built from
 predictive modeling tools, but the created model must be sufficiently
 interpretable to provide a reason for any adverse actions (rejections).

 Content
 =======
 The Home Equity dataset (HMEQ) contains baseline and loan performance
 information for 5,960 recent home equity loans. The target (BAD) is a binary
 variable indicating whether an applicant eventually defaulted or was
 seriously delinquent. This adverse outcome occurred in 1,189 cases (20%). For
 each applicant, 12 input variables were recorded.

 Attributes
 ==========
 Name    Description

 TARGET  Label: 1 = client defaulted on loan - 0 = loan repaid

 LOAN    Amount of the loan request

 MORTDUE Amount due on existing mortgage

 VALUE   Value of current property

 YOJ     Years at present job

 DEROG   Number of major derogatory reports

 DELINQ  Number of delinquent credit lines

 CLAGE   Age of oldest trade line in months

 NINQ    Number of recent credit lines

 CLNO    Number of credit lines

 DEBTINC Debt-to-income ratio



In [ ]:
print( pd.read_csv("https://raw.githubusercontent.com/DiploDatos/IntroduccionAprendizajeAutomatico/master/data/loan_data.csv", comment="#"))



# Modelo lineal con SGDClassifier

En esta parte vamos a usar un clasificador lineal entrenado mediante descenso de gradiente estocástico.

Antes de evaluar resultados, respondamos:

1. ¿Qué significa que sea un modelo lineal?
2. ¿Qué función de pérdida estamos usando?
3. ¿Qué hiperparámetros aparecen en la documentación?
4. ¿Qué valores toma el modelo por defecto?
5. ¿Por qué la tasa de aprendizaje y la regularización pueden afectar el resultado?

Esto conecta directamente con la clase teórica sobre función de costo, optimización y descenso de gradiente.


## Ejercicio 2: Predicción con Modelos Lineales

En este ejercicio se entrenarán modelos lineales de clasificación para predecir la variable objetivo.

Para ello, deberán utilizar la clase SGDClassifier de scikit-learn.

Documentación:
- https://scikit-learn.org/stable/modules/sgd.html
- https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html


### Ejercicio 2.1: SGDClassifier con hiperparámetros por defecto

Entrenar y evaluar el clasificador SGDClassifier usando los valores por omisión de scikit-learn para todos los parámetros. Únicamente **fijar la semilla aleatoria** para hacer repetible el experimento.

Evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión

# Búsqueda de hiperparámetros con validación cruzada

Ahora no queremos quedarnos con una sola configuración del modelo.

Vamos a probar varias combinaciones de hiperparámetros y evaluarlas mediante validación cruzada.

Cuando analicemos los resultados de `GridSearchCV`, no miremos solamente el mejor score. También observemos:

1. `best_params_`: mejor combinación de hiperparámetros.
2. `best_score_`: mejor desempeño promedio en validación cruzada.
3. `mean_test_score`: media del score en validación.
4. `std_test_score`: variabilidad entre folds.
5. `mean_train_score`: desempeño promedio en entrenamiento, si está disponible.

Una configuración con media alta y desviación estándar baja suele ser más estable que una configuración con media alta pero gran variabilidad.


### Ejercicio 2.2: Ajuste de Hiperparámetros

Seleccionar valores para los hiperparámetros principales del SGDClassifier. Como mínimo, probar diferentes funciones de loss, tasas de entrenamiento y tasas de regularización.

Para ello, usar grid-search y 5-fold cross-validation sobre el conjunto de entrenamiento para explorar muchas combinaciones posibles de valores.

Reportar accuracy promedio y varianza para todas las configuraciones.

Para la mejor configuración encontrada, evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión

Documentación:
- https://scikit-learn.org/stable/modules/grid_search.html
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

# Árbol de decisión

Ahora vamos a repetir el análisis usando un árbol de decisión.

Este modelo tiene una interpretación diferente al clasificador lineal:

- divide el espacio de atributos mediante reglas;
- puede capturar relaciones no lineales;
- puede sobreajustar si crece demasiado.

Preguntas orientadoras:

1. ¿Qué profundidad alcanza el árbol por defecto?
2. ¿Hay evidencia de sobreajuste?
3. ¿Qué hiperparámetros podemos ajustar?
4. ¿Qué criterio conviene probar: `gini`, `entropy` o `log_loss`?
5. ¿Qué efecto tiene `max_depth`?
6. ¿Qué efecto tiene `min_samples_leaf`?


## Ejercicio 3: Árboles de Decisión

En este ejercicio se entrenarán árboles de decisión para predecir la variable objetivo.

Para ello, deberán utilizar la clase DecisionTreeClassifier de scikit-learn.

Documentación:
- https://scikit-learn.org/stable/modules/tree.html
  - https://scikit-learn.org/stable/modules/tree.html#tips-on-practical-use
- https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
- https://scikit-learn.org/stable/auto_examples/tree/plot_unveil_tree_structure.html

### Ejercicio 3.1: DecisionTreeClassifier con hiperparámetros por defecto

Entrenar y evaluar el clasificador DecisionTreeClassifier usando los valores por omisión de scikit-learn para todos los parámetros. Únicamente **fijar la semilla aleatoria** para hacer repetible el experimento.

Evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión


### Ejercicio 3.2: Ajuste de Hiperparámetros

Seleccionar valores para los hiperparámetros principales del DecisionTreeClassifier. Como mínimo, probar diferentes criterios de partición (criterion), profundidad máxima del árbol (max_depth), y cantidad mínima de samples por hoja (min_samples_leaf).

Para ello, usar grid-search y 5-fold cross-validation sobre el conjunto de entrenamiento para explorar muchas combinaciones posibles de valores.

Reportar accuracy promedio y varianza para todas las configuraciones.

Para la mejor configuración encontrada, evaluar sobre el conjunto de **entrenamiento** y sobre el conjunto de **evaluación**, reportando:
- Accuracy
- Precision
- Recall
- F1
- matriz de confusión


Documentación:
- https://scikit-learn.org/stable/modules/grid_search.html
- https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

# Conclusión final del trabajo práctico

Para cerrar el TP, escribamos una conclusión breve respondiendo:

1. ¿Cuál fue el mejor modelo encontrado?
2. ¿Con qué hiperparámetros?
3. ¿Qué métrica usamos para decidir?
4. ¿Por qué esa métrica es adecuada para este problema?
5. ¿El modelo parece estable entre folds?
6. ¿Hay señales de sobreajuste?
7. ¿Qué tipo de error nos preocupa más: falso positivo o falso negativo?
8. ¿Recomendaríamos usar este modelo en un contexto real? ¿Qué advertencias haríamos?

La respuesta no debería limitarse a copiar números. Debe justificar la decisión usando las métricas y el significado del problema.


# Sugerencia opcional: tabla comparativa final

Podemos resumir los resultados en una tabla como esta:

| Modelo | Mejor configuración | Accuracy test | Precision | Recall | F1 | Comentario |
|---|---|---:|---:|---:|---:|---|
| SGDClassifier | ... | ... | ... | ... | ... | ... |
| DecisionTreeClassifier | ... | ... | ... | ... | ... | ... |

Esta tabla ayuda a comparar modelos de manera ordenada.
